# 🎟️ TICKETMELT — Final A100 Training Notebook

**Runtime:** A100 GPU (Colab Pro) | **Estimated time: ~75 min end to end**

| Cell | What it does | Time |
|------|-------------|------|
| 1 | Setup + install | 4 min |
| 2 | Clone + patches + verify | 1 min |
| 3 | Load model | 2 min |
| 4 | Prompt + inference test | 1 min |
| 5 | Baseline eval (20 eps) | 6 min |
| 6 | GRPO training (128 eps, 16 steps) | ~55 min |
| 7 | Post-eval + plots + save | 10 min |

**Run in order. Do not skip. Do not re-run model cell.**

In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────────
import warnings, logging, os, subprocess, sys, json, time, shutil
from pathlib import Path
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('unsloth').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
assert torch.cuda.is_available(), 'No GPU found — change runtime'
gpu = torch.cuda.get_device_properties(0)
vram = gpu.total_memory / 1e9
print(f'✅ GPU: {gpu.name} | VRAM: {vram:.1f} GB')
assert vram > 20, f'This notebook needs A100 (40GB). You have {vram:.1f}GB. Use the lightweight notebook instead.'

!pip install -q --upgrade pip
!pip install -q 'unsloth[colab-new]' 2>&1 | tail -1
!pip install -q 'trl>=0.12' datasets matplotlib 2>&1 | tail -1
print('✅ Cell 1 done')

In [ ]:
# ── CELL 2: Fresh clone + all patches ─────────────────────────────────────────
# Single source of truth. Everything configured here.

HF_REPO = 'https://huggingface.co/spaces/TheAllanB/ticketmelt'

# Always fresh clone — no stale state
if os.path.exists('ticketmelt'):
    shutil.rmtree('ticketmelt')
subprocess.run(['git', 'clone', HF_REPO, 'ticketmelt'],
               check=True, capture_output=True)
sys.path.insert(0, 'ticketmelt')
print('✅ Fresh clone')

# Verify env values from source
import re
with open('ticketmelt/src/rewards.py') as f:
    _src = f.read()
assert '0.45' in _src, '❌ GOODNESS_THRESHOLD not 0.45 in repo — push fix first'
assert 'return 0.0  # never deployed' in _src, '❌ R3 fix not in repo — push fix first'
print('✅ Env code verified (threshold=0.45, R3 fixed)')

# Fresh module imports
import importlib
import src.rewards, src.environment
importlib.reload(src.rewards)
importlib.reload(src.environment)

from src.environment import TicketmeltEnv
from src.models import Action
import src.rewards as rw
import src.environment as env_mod
from src.opponents import DUMB_PEERS  # all peers → PROD_A = clear signal

# ── PATCH 1: R3 fix (in case repo version differs) ───────────────────────────
def _r3(state):
    me = state.trained_engineer
    deploys, cols = [], []
    for rec in state.history:
        c = rec.commitments.get(me)
        if c in ('DEPLOY_PROD_A', 'DEPLOY_PROD_B'):
            deploys.append(rec)
            if f'PROD_{c[-1]}' in rec.collisions:
                cols.append(rec)
    return 0.0 if not deploys else 1.0 - len(cols) / len(deploys)
rw.reward_r3_clean_deploys = _r3

# ── PATCH 2: Inline reward (bypasses env threshold entirely) ─────────────────
# We compute binary reward ourselves — no dependency on env config.
# r1 > 0.5 = own service restored. weighted > 0.35 = minimal group welfare.
def compute_inline_reward(bd):
    r1 = bd.get('r1_service_restored', 0)
    r2 = bd.get('r2_site_uptime', 0)
    r3 = bd.get('r3_clean_deploys', 0)
    r4 = bd.get('r4_yield_to_critical', 0)
    w = 0.5*r1 + 0.3*r2 + 0.1*r3 + 0.1*r4
    return 1.0 if (r1 > 0.5 and w > 0.35) else 0.0, r1, r2, r3, r4, w

# ── PATCH 3: Force DUMB peers on every reset ──────────────────────────────────
_orig_reset = env_mod.TicketmeltEnv.reset
def _reset_with_dumb(self, seed=None):
    self.peer_strategies = DUMB_PEERS
    return _orig_reset(self, seed=seed)
env_mod.TicketmeltEnv.reset = _reset_with_dumb

# ── VERIFY: always-PROD_B should win 60-90% ──────────────────────────────────
wins = []
for s in range(10):
    e = TicketmeltEnv(seed=s)
    obs = e.reset()
    for _ in range(8):
        _, _, done, info = e.step(Action(commitment='DEPLOY_PROD_B', channel_msg=''))
        if done:
            bd = info.get('reward_breakdown', {})
            reward, *_ = compute_inline_reward(bd)
            wins.append(reward)
            break
wr = sum(wins) / len(wins)
print(f'Always-PROD_B win rate: {wr:.0%}')
assert 0.4 <= wr <= 0.95, f'Win rate {wr:.0%} out of expected range — check patches'
print('✅ Cell 2 done — signal is clean')

In [ ]:
# ── CELL 3: Load model (run ONCE per session) ──────────────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=1024,   # 1024 is enough for our short prompts, faster than 2048
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
FastLanguageModel.for_inference(model)
used = torch.cuda.memory_allocated() / 1e9
print(f'✅ Model ready | VRAM: {used:.1f} / {vram:.1f} GB')
print(f'   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print('✅ Cell 3 done')

In [ ]:
# ── CELL 4: Prompt + inference functions ───────────────────────────────────────

# Explicit: tells model peers go to PROD_A → use PROD_B
# This is the exact prompt that worked in the lightweight run
SYSTEM_PROMPT = """You are an ATC controller managing approach sequencing.

IMPORTANT: The other 3 controllers almost always clear aircraft to RWY-A.
If you also clear to RWY-A, you will cause a RUNWAY COLLISION and waste the round.
Clear to RWY-B to avoid collisions and land your aircraft safely.

Rules:
- Use DEPLOY_PROD_B (clear to RWY-B) unless you see clear evidence others are going there
- Use MONITOR (HOLD) only if your aircraft has already landed
- You have limited fuel rounds — every wasted round risks losing the aircraft

Respond ONLY with JSON (nothing else):
{"channel_msg": "brief ATC status", "commitment": "DEPLOY_PROD_B"}"""

def obs_to_prompt(obs):
    svc = obs.my_service
    lines = [
        f'ROUND {obs.current_round+1}/{obs.total_rounds}',
        f'YOUR SERVICE: {svc.name.upper()}{" [URGENT]" if svc.urgency_flag else ""}',
        f'Rounds needed: {svc.fix_rounds_remaining} | Deadline: {svc.deadline_round}',
        'PEERS:',
    ]
    for name, peer in obs.peer_progress.items():
        s = 'DONE' if peer['completed'] else f"{peer['rounds_remaining']} left"
        lines.append(f'  {peer["service"].upper()}: {s}')
    if obs.history:
        lines.append('CHANNEL:')
        for rec in obs.history[-2:]:
            for eng, msg in rec.messages.items():
                if msg: lines.append(f'  [{eng}]: {msg}')
            if rec.collisions:
                lines.append(f'  COLLISION on {",".join(rec.collisions)}')
            for eng, srv in rec.successful_deploys.items():
                lines.append(f'  {eng} deployed to {srv}')
    lines.append('JSON: {"channel_msg": "...", "commitment": "DEPLOY_PROD_B"}')
    return '\n'.join(lines)

def parse_action(text):
    VALID = ('DEPLOY_PROD_A', 'DEPLOY_PROD_B', 'MONITOR')
    try:
        clean = text.strip()
        if '```' in clean:
            clean = clean.split('```')[1].lstrip('json')
        s, e = clean.find('{'), clean.rfind('}') + 1
        if s < 0 or e <= 0:
            return Action(commitment='DEPLOY_PROD_B', channel_msg='')
        data = json.loads(clean[s:e])
        c = data.get('commitment', 'DEPLOY_PROD_B')
        if c not in VALID: c = 'DEPLOY_PROD_B'  # default to B, never MONITOR
        return Action(commitment=c, channel_msg=str(data.get('channel_msg', ''))[:100])
    except Exception:
        return Action(commitment='DEPLOY_PROD_B', channel_msg='')

def generate_action(obs, temperature=0.8):
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_to_prompt(obs)},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_new_tokens=60,  # short = fast; enough for JSON
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return raw, parse_action(raw)

# Quick test — want ≥5/8 deploying (not MONITOR)
deploy_count = 0
for seed in range(8):
    e = TicketmeltEnv(seed=seed)
    _, action = generate_action(e.reset())
    is_deploy = action.commitment in ('DEPLOY_PROD_A', 'DEPLOY_PROD_B')
    deploy_count += int(is_deploy)
    print(f'  seed {seed}: {action.commitment}')

print(f'\nDeploy rate: {deploy_count}/8 '
      f'{"✅" if deploy_count >= 5 else "⚠️ low — parse_action defaults to PROD_B so training will still work"}')
print('✅ Cell 4 done')

In [ ]:
# ── CELL 5: BASELINE EVAL — run ONCE, before training ─────────────────────────
# These numbers are your README 'before' column.
# Do NOT re-run this cell after training starts.

import json as jsonlib
Path('plots').mkdir(exist_ok=True)

EVAL_SEEDS = list(range(20))  # fixed seeds — same used for post-eval comparison

def run_episode(seed, temperature=0.8):
    ep_env = TicketmeltEnv(seed=seed)
    obs = ep_env.reset()
    done = False
    transcript = []
    while not done:
        raw, action = generate_action(obs, temperature)
        obs, _, done, info = ep_env.step(action)
        transcript.append({
            'round': obs.current_round,
            'commitment': action.commitment,
            'msg': action.channel_msg,
        })
    bd = info.get('reward_breakdown', {})
    sm = info.get('episode_summary', {})
    reward, r1, r2, r3, r4, w = compute_inline_reward(bd)
    return {
        'seed': seed, 'reward': reward,
        'r1': r1, 'r2': r2, 'r3': r3, 'r4': r4, 'weighted': w,
        'collision_rate': sm.get('total_collisions', 0) / max(1, sm.get('rounds_played', 1)),
        'transcript': transcript,
    }

print(f'Baseline eval ({len(EVAL_SEEDS)} episodes)...')
t0 = time.time()
baseline_results = []
for i, seed in enumerate(EVAL_SEEDS):
    baseline_results.append(run_episode(seed))
    if (i + 1) % 5 == 0:
        wr = sum(r['reward'] for r in baseline_results) / len(baseline_results)
        print(f'  {i+1}/{len(EVAL_SEEDS)} | win={wr:.0%} | {time.time()-t0:.0f}s', end='\r')
print()

avg = lambda k: sum(r[k] for r in baseline_results) / len(baseline_results)
baseline_summary = {k: avg(k) for k in ['reward','r1','r2','r3','r4','weighted','collision_rate']}

print('\nBASELINE RESULTS:')
for k, v in baseline_summary.items():
    print(f'  {k:<15}: {v:.3f}')

with open('baseline_metrics.json', 'w') as f:
    jsonlib.dump([{k: v for k, v in r.items() if k != 'transcript'}
                  for r in baseline_results], f)

wr = baseline_summary['reward']
print(f'\nwin_rate = {wr:.2f}')
if wr > 0.75:
    print('⚠️  High baseline — model already good. Training may show small delta. Proceed anyway.')
elif wr < 0.15:
    print('⚠️  Low baseline — model rarely wins. Training needs more episodes or check patches.')
else:
    print('✅ Good baseline range — proceed to training')
print(f'\n📋 Save these numbers — they are your README baseline column')
print('✅ Cell 5 done')

In [ ]:
# ── CELL 6: GRPO TRAINING ──────────────────────────────────────────────────────
# 128 episodes = 16 steps ≈ 55-65 min on A100
# training_log accumulates — re-runnable if you want more steps

from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

N_TRAIN_EPISODES = 128  # 128 / 8 effective batch = 16 steps

if 'training_log' not in globals():
    globals()['training_log'] = []
if '_seed_offset' not in globals():
    globals()['_seed_offset'] = 1000  # never overlap with eval seeds 0-19

TRAIN_SEEDS = list(range(_seed_offset, _seed_offset + N_TRAIN_EPISODES))
globals()['_seed_offset'] += N_TRAIN_EPISODES

# ── Reward function ────────────────────────────────────────────────────────────
def grpo_reward_fn(completions, prompts, **kwargs):
    seeds = kwargs.get('seed', [None] * len(completions))
    if not isinstance(seeds, (list, tuple)):
        seeds = [seeds] * len(completions)
    rewards = []
    for idx, (comp, prompt) in enumerate(zip(completions, prompts)):
        seed = int(seeds[idx]) if seeds[idx] is not None else hash(prompt) % 10000
        ep_env = TicketmeltEnv(seed=seed)
        obs = ep_env.reset()
        action = parse_action(comp)
        done = False
        bd = {}
        while not done:
            obs, _, done, info = ep_env.step(action)
            if not done:
                _, action = generate_action(obs)
            bd = info.get('reward_breakdown', {})
        final, r1, r2, r3, r4, w = compute_inline_reward(bd)
        rewards.append(final)
        training_log.append({'reward': final, 'r1': r1, 'r2': r2, 'r3': r3, 'r4': r4})
    return rewards

# ── Dataset ────────────────────────────────────────────────────────────────────
def make_prompt(seed):
    e = TicketmeltEnv(seed=seed)
    msgs = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_to_prompt(e.reset())},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

train_data = Dataset.from_dict({
    'prompt': [make_prompt(s) for s in TRAIN_SEEDS],
    'seed':   [int(s) for s in TRAIN_SEEDS],
})

# ── Config ─────────────────────────────────────────────────────────────────────
# Batch math: per_device=2 × grad_accum=4 = effective batch 8
# num_generations=8: at 40-60% baseline, ~4-5 wins per batch → clean gradient
config = GRPOConfig(
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    num_generations=8,               # 8 completions per prompt
    temperature=0.9,
    max_grad_norm=0.1,
    optim='adamw_8bit',
    logging_steps=1,
    output_dir='checkpoints',
    save_steps=999,                  # no mid-run saves — faster
    report_to='none',
    max_prompt_length=800,           # our prompts are short
    max_completion_length=60,        # enough for JSON action
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_fn,
    args=config,
    train_dataset=train_data,
    processing_class=tokenizer,
)

steps = N_TRAIN_EPISODES // (config.per_device_train_batch_size * config.gradient_accumulation_steps)
print(f'🚀 Training')
print(f'   Episodes : {N_TRAIN_EPISODES} (seeds {TRAIN_SEEDS[0]}–{TRAIN_SEEDS[-1]})')
print(f'   Steps    : {steps}')
print(f'   Est. time: ~{steps * 4} min on A100')
print(f'   training_log entries so far: {len(training_log)}')
print(f'\n   Watch first log line — reward should be >0.0\n')

t0 = time.time()
trainer.train()
elapsed = (time.time() - t0) / 60

print(f'\n✅ Training done in {elapsed:.0f} min')
print(f'   training_log entries: {len(training_log)}')
if training_log:
    r = [e['reward'] for e in training_log]
    print(f'   Overall win rate : {sum(r)/len(r):.0%}')
    print(f'   Last 20 win rate : {sum(r[-20:])/min(20,len(r)):.0%}')
    print(f'   Non-zero entries : {sum(1 for x in r if x > 0)}/{len(r)}')

In [ ]:
# ── CELL 7: POST-EVAL + PLOTS + SAVE + PUSH ────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib
import json as jsonlib
matplotlib.rcParams['figure.dpi'] = 150

# ── Post-training eval ─────────────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

print(f'Post-eval ({len(EVAL_SEEDS)} episodes, same seeds as baseline)...')
t0 = time.time()
trained_results = []
for i, seed in enumerate(EVAL_SEEDS):
    trained_results.append(run_episode(seed))
    if (i + 1) % 5 == 0:
        wr = sum(r['reward'] for r in trained_results) / len(trained_results)
        print(f'  {i+1}/{len(EVAL_SEEDS)} | win={wr:.0%} | {time.time()-t0:.0f}s', end='\r')
print()

avg = lambda k: sum(r[k] for r in trained_results) / len(trained_results)
trained_summary = {k: avg(k) for k in ['reward','r1','r2','r3','r4','weighted','collision_rate']}

with open('trained_metrics.json', 'w') as f:
    jsonlib.dump([{k: v for k, v in r.items() if k != 'transcript'}
                  for r in trained_results], f)

# ── Comparison table ───────────────────────────────────────────────────────────
print(f'\n{"="*58}')
print(f'{"METRIC":<18} {"BASELINE":>10} {"TRAINED":>10} {"DELTA":>10}')
print(f'{"="*58}')
for k in ['reward','r1','r2','r3','r4','collision_rate']:
    b = baseline_summary.get(k, 0)
    t = trained_summary.get(k, 0)
    arrow = '↑' if t - b > 0.02 else ('↓' if t - b < -0.02 else '→')
    print(f'  {k:<16} {b:>10.3f} {t:>10.3f}  {arrow} {t-b:>+.3f}')
print(f'{"="*58}')
print('\n📋 Copy this table into your README')

# ── Helper ─────────────────────────────────────────────────────────────────────
def rolling(vals, w=15):
    return [sum(vals[max(0,i-w+1):i+1]) / len(vals[max(0,i-w+1):i+1])
            for i in range(len(vals))]

# ── Plot 1: Reward curve ───────────────────────────────────────────────────────
rewards = [e['reward'] for e in training_log]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards, alpha=0.15, color='steelblue', linewidth=0.8)
ax.plot(rolling(rewards), color='steelblue', linewidth=2, label='Rolling mean (window=15)')
ax.axhline(baseline_summary['reward'], color='red', linestyle='--', linewidth=1.5,
           label=f'Baseline ({baseline_summary["reward"]:.0%})')
ax.set_xlabel('Training Episode')
ax.set_ylabel('Reward (0=fail, 1=success)')
ax.set_title('TICKETMELT — GRPO Training Reward Curve', fontweight='bold')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ plots/reward_curve.png')

# ── Plot 2: Per-component rewards ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for key, label, color in [
    ('r1', 'R1: Service Restored', '#2196F3'),
    ('r2', 'R2: Site Uptime',      '#4CAF50'),
    ('r3', 'R3: No Collision',     '#FF9800'),
    ('r4', 'R4: Yield to Critical','#9C27B0'),
]:
    vals = [e[key] for e in training_log]
    ax.plot(rolling(vals), color=color, linewidth=2, label=label)
ax.set_xlabel('Training Episode')
ax.set_ylabel('Component Value (rolling mean)')
ax.set_title('TICKETMELT — Per-Component Rewards During Training', fontweight='bold')
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/component_rewards.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ plots/component_rewards.png')

# ── Plot 3: Before vs after bar chart ─────────────────────────────────────────
metrics = [
    ('reward', 'Win Rate'),
    ('r1', 'R1: Service'),
    ('r2', 'R2: Uptime'),
    ('r3', 'R3: No Collision'),
    ('r4', 'R4: Yield'),
]
x = range(len(metrics))
fig, ax = plt.subplots(figsize=(10, 5))
bb = ax.bar([i - 0.2 for i in x],
            [baseline_summary[m[0]] for m in metrics], 0.35,
            label='Baseline (untrained)', color='#EF5350', alpha=0.85)
bt = ax.bar([i + 0.2 for i in x],
            [trained_summary[m[0]] for m in metrics], 0.35,
            label='After GRPO training', color='#42A5F5', alpha=0.85)
for bar in list(bb) + list(bt):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(list(x))
ax.set_xticklabels([m[1] for m in metrics])
ax.set_ylabel('Score (0–1)')
ax.set_ylim(0, 1.15)
ax.set_title('TICKETMELT — Baseline vs Trained Model', fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ plots/before_after.png')

# ── Save adapter ───────────────────────────────────────────────────────────────
model.save_pretrained('ticketmelt-lora-adapter')
tokenizer.save_pretrained('ticketmelt-lora-adapter')
print('✅ LoRA adapter saved')

# ── Copy artifacts into repo ───────────────────────────────────────────────────
os.makedirs('ticketmelt/plots', exist_ok=True)
os.makedirs('ticketmelt/examples', exist_ok=True)
for fname in ['reward_curve.png', 'component_rewards.png', 'before_after.png']:
    shutil.copy(f'plots/{fname}', f'ticketmelt/plots/{fname}')
shutil.copy('baseline_metrics.json', 'ticketmelt/baseline_metrics.json')
shutil.copy('trained_metrics.json',  'ticketmelt/trained_metrics.json')

# ── Save before/after transcript examples ─────────────────────────────────────
def fmt_transcript(r, label):
    lines = [
        f'=== {label} (seed={r["seed"]}) ===',
        f'Result: {"WIN" if r["reward"]==1.0 else "FAIL"}  '
        f'Collision rate: {r["collision_rate"]:.0%}',
        ''
    ]
    for t in r['transcript']:
        msg_part = f'[{t["msg"]}] ' if t['msg'] else ''
        lines.append(f'  Round {t["round"]}: {msg_part}→ {t["commitment"]}')
    return '\n'.join(lines)

b_map = {r['seed']: r for r in baseline_results}
t_map = {r['seed']: r for r in trained_results}

# Best examples: baseline fails, trained wins
interesting = [s for s in EVAL_SEEDS
               if b_map[s]['reward'] == 0.0 and t_map[s]['reward'] == 1.0]
picks = interesting[:3] if len(interesting) >= 3 else EVAL_SEEDS[:3]
print(f'\nBest demo seeds (baseline❌ trained✅): {interesting[:5]}')

for i, seed in enumerate(picks):
    content = (fmt_transcript(b_map[seed], 'BASELINE (untrained)')
               + '\n\n' + '='*50 + '\n\n'
               + fmt_transcript(t_map[seed], 'TRAINED MODEL'))
    path = f'ticketmelt/examples/example_{i+1}_seed{seed}.txt'
    with open(path, 'w') as f:
        f.write(content)
    print(f'✅ {path}')

# ── Git commit and push ────────────────────────────────────────────────────────
os.chdir('ticketmelt')
subprocess.run(['git', 'add', '.'], check=True)
subprocess.run(['git', 'commit', '-m',
                'Add training results: plots, metrics, examples'], check=True)
subprocess.run(['git', 'push', 'hf', 'main'], check=True)
os.chdir('..')
print('\n✅ Pushed to HF Space')

# ── Final checklist ────────────────────────────────────────────────────────────
print('\n🏁 REMAINING TASKS:')
for item in [
    'Fill README placeholders: Space URL, Colab URL, results table, plot images',
    'Record 2-min video using ATC visualization + real transcript from examples/',
    'Write HF blog post (can be shortened README)',
    'Submit HF Space URL (not GitHub)',
]:
    print(f'  ☐ {item}')

## If Training Completes But You Want More Steps

Just re-run Cell 6. Seeds auto-advance, `training_log` accumulates, plots in Cell 7 will include all episodes.

## Hourly check (paste in new cell)
```python
if training_log:
    r = [e['reward'] for e in training_log]
    print(f'Episodes: {len(r)} | Overall: {sum(r)/len(r):.0%} | Last 20: {sum(r[-20:])/min(20,len(r)):.0%}')
```

## What healthy training looks like
- `reward` fluctuates between 0.0 and 1.0 — normal
- `kl` stays below 0.01 throughout — healthy
- Rolling mean trends above baseline dashed line by step 10+
- `Non-zero entries` at end should be >40% of total